# Azzaro: Whisper turbo vs small vs large

Comparamos tres modelos de Whisper sobre el mismo video, contamos diferencias de palabras y revisamos los clips donde no coinciden. El caso puntual a mirar es `racingista` / `racinguista`.

## 1. Configuracion

In [ ]:
from itertools import combinations
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Markdown, Video, display

from evaluation.src.whisper_model_comparison import (
    alinear_diferencias,
    descargar_video_yt,
    exportar_clips_diferencias,
    extraer_palabras,
    normalizar_texto,
    transcribir_whisper,
)

VIDEO_URL = "https://www.youtube.com/watch?v=a4ggqJZXnQE"
VIDEO_PATH = None  # si ya tenes el mp4 local, poner aca la ruta y dejar VIDEO_URL como esta

MODELOS = ["turbo", "small", "large"]
COOKIES_FROM_BROWSER = "chrome"  # poner None si YouTube no pide login
FORCE_TRANSCRIBE = False

CLIP_MARGIN_SECONDS = 0.25  # margen visual alrededor de la diferencia; subir si falta contexto
MAX_CASOS_A_MOSTRAR = None  # poner un numero si queres limitar la revision visual

WORKDIR = ROOT / "evaluation" / "outputs" / "azzaro_whisper"
WORKDIR


## 2. Descargar o cargar video

In [ ]:
if VIDEO_PATH:
    video_path = Path(VIDEO_PATH).expanduser().resolve()
else:
    video_path = descargar_video_yt(
        VIDEO_URL,
        WORKDIR / "videos",
        nombre_base="azzaro_racing_caracas",
        cookies_from_browser=COOKIES_FROM_BROWSER,
    )

video_path

## 3. Transcribir con turbo, small y large

Esto cachea cada JSON en `evaluation/outputs/azzaro_whisper/transcripts/`. Si ya existe, no vuelve a transcribir.

In [ ]:
resultados = {}
palabras = {}

for model_name in MODELOS:
    print(f"Transcribiendo / cargando cache: {model_name}")
    resultados[model_name] = transcribir_whisper(
        video_path,
        model_name=model_name,
        output_dir=WORKDIR / "transcripts",
        force=FORCE_TRANSCRIBE,
    )
    palabras[model_name] = extraer_palabras(resultados[model_name], model_name)

pd.DataFrame([
    {"modelo": modelo, "palabras": len(palabras_modelo)}
    for modelo, palabras_modelo in palabras.items()
])

## 4. Detectar diferencias donde difieren los tres modelos

A partir de aca filtramos fuerte: solo quedan tramos donde `turbo`, `small` y `large` producen tres textos distintos.


In [ ]:
def texto_modelo_en_rango(palabras_modelo, start, end, margen=0.20):
    seleccionadas = []
    left = max(float(start) - margen, 0.0)
    right = float(end) + margen
    for palabra in palabras_modelo:
        p_start = float(palabra.get("start", 0.0))
        p_end = float(palabra.get("end", p_start))
        centro = (p_start + p_end) / 2
        if left <= centro <= right or (p_start < right and p_end > left):
            seleccionadas.append(str(palabra.get("word", "")).strip())
    return " ".join(tok for tok in seleccionadas if tok).strip()


def unir_intervalos(intervalos, gap=0.75):
    intervalos = sorted(intervalos, key=lambda x: (x[0], x[1]))
    unidos = []
    for start, end in intervalos:
        if not unidos or start > unidos[-1][1] + gap:
            unidos.append([float(start), float(end)])
        else:
            unidos[-1][1] = max(unidos[-1][1], float(end))
    return unidos


# Se alinean todos los pares, pero esto queda como paso interno.
# La tabla final NO muestra diferencias por pares: solo casos donde los tres difieren.
diferencias_por_par = []
for modelo_a, modelo_b in combinations(MODELOS, 2):
    diferencias = alinear_diferencias(
        palabras[modelo_a],
        palabras[modelo_b],
        contexto=5,
    )
    for diferencia in diferencias:
        diferencias_por_par.append({
            "start": float(diferencia["start"]),
            "end": float(diferencia["end"]),
            "comparacion": f"{modelo_a}_vs_{modelo_b}",
        })

intervalos_diff = [
    (d["start"], d["end"])
    for d in diferencias_por_par
    if d["end"] > d["start"]
]

filas_tres_modelos = []
for start, end in unir_intervalos(intervalos_diff):
    textos = {
        modelo: texto_modelo_en_rango(palabras[modelo], start, end)
        for modelo in MODELOS
    }
    textos_norm = {modelo: normalizar_texto(texto) for modelo, texto in textos.items()}

    if all(textos_norm.values()) and len(set(textos_norm.values())) == len(MODELOS):
        filas_tres_modelos.append({
            "diff_id": len(filas_tres_modelos) + 1,
            "start": round(start, 3),
            "end": round(end, 3),
            "texto_turbo": textos["turbo"],
            "texto_small": textos["small"],
            "texto_large": textos["large"],
            "norm_turbo": textos_norm["turbo"],
            "norm_small": textos_norm["small"],
            "norm_large": textos_norm["large"],
        })

filas_tres_modelos = exportar_clips_diferencias(
    video_path,
    filas_tres_modelos,
    output_dir=WORKDIR / "clips_tres_modelos",
    margen=CLIP_MARGIN_SECONDS,
    max_clips=len(filas_tres_modelos),
)

df_tres = pd.DataFrame(filas_tres_modelos)
if not df_tres.empty and "review_index" not in df_tres.columns:
    df_tres.insert(0, "review_index", range(len(df_tres)))

df_tres.to_csv(WORKDIR / "diferencias_tres_modelos.csv", index=False)
print(f"Casos donde difieren los tres modelos: {len(df_tres)}")

cols = ["review_index", "start", "end", "clip_start", "clip_end", "texto_turbo", "texto_small", "texto_large", "clip_path"]
display(df_tres[cols] if not df_tres.empty else df_tres)


## 5. Revisar caso por caso

Cada caso muestra el tramo exacto donde difieren los tres modelos y el clip exportado. Si el video dice un poco mas, es por el margen visual alrededor del tramo.


In [ ]:
if df_tres.empty:
    print("No hay casos donde turbo, small y large tengan tres textos distintos.")
else:
    filas = df_tres if MAX_CASOS_A_MOSTRAR is None else df_tres.head(MAX_CASOS_A_MOSTRAR)
    for _, row in filas.iterrows():
        display(Markdown(
            f"### Caso {int(row['review_index'])} | "
            f"diferencia {row['start']}s-{row['end']}s | "
            f"clip {row['clip_start']}s-{row['clip_end']}s"
        ))

        print("TRAMO EXACTO DONDE DIFIEREN")
        print("turbo:", row["texto_turbo"])
        print("small:", row["texto_small"])
        print("large:", row["texto_large"])

        print("\nTEXTO DEL CLIP COMPLETO, INCLUYENDO MARGEN")
        for modelo in MODELOS:
            texto_clip = texto_modelo_en_rango(
                palabras[modelo],
                row["clip_start"],
                row["clip_end"],
                margen=0.0,
            )
            print(f"{modelo}:", texto_clip)

        display(Video(row["clip_path"], embed=True, html_attributes="controls"))


In [ ]:
# Opcional: guardar una decision manual para un caso puntual.
REVIEW_INDEX = 0
MODELO_CORRECTO = ""  # opciones: "turbo", "small", "large", "ninguno", "empate"
NOTA = ""

revision_path = WORKDIR / "revision_manual_tres_modelos.csv"

if df_tres.empty:
    print("No hay casos de tres modelos para guardar.")
else:
    if revision_path.exists():
        revision_tres = pd.read_csv(revision_path)
    else:
        revision_tres = df_tres.copy()
        revision_tres["modelo_correcto"] = ""
        revision_tres["nota"] = ""

    if MODELO_CORRECTO:
        mask = revision_tres["review_index"].eq(REVIEW_INDEX)
        revision_tres.loc[mask, "modelo_correcto"] = MODELO_CORRECTO
        revision_tres.loc[mask, "nota"] = NOTA
        revision_tres.to_csv(revision_path, index=False)
        display(revision_tres.loc[mask, ["review_index", "texto_turbo", "texto_small", "texto_large", "modelo_correcto", "nota"]])
        print(f"Guardado en: {revision_path}")
    else:
        completadas = revision_tres["modelo_correcto"].fillna("").ne("").sum()
        print(f"Completa MODELO_CORRECTO para guardar. Revisadas: {completadas} / {len(revision_tres)}")
        display(revision_tres[["review_index", "texto_turbo", "texto_small", "texto_large", "modelo_correcto", "nota"]])
